In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor  # stronger regressor




In [2]:
train_path = "../input/ventilator-pressure-prediction/train.csv"
test_path = "../input/ventilator-pressure-prediction/test.csv"

train_cols = ["R", "C", "time_step", "u_in", "u_out", "breath_id", "pressure"]
test_cols = ["R", "C", "time_step", "u_in", "u_out", "breath_id", "id"]

dtype_train = {
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "uint8",
    "breath_id": "int32",
    "pressure": "float32",
}
dtype_test = {
    "R": "int16",
    "C": "int16",
    "time_step": "float32",
    "u_in": "float32",
    "u_out": "uint8",
    "breath_id": "int32",
    "id": "int32",
}

train_df = pd.read_csv(train_path, usecols=train_cols, dtype=dtype_train)
test_df = pd.read_csv(test_path, usecols=test_cols, dtype=dtype_test)

# Ensure no missing pressure values
train_df = train_df.dropna(subset=["pressure"]).reset_index(drop=True)

feature_cols = ["R", "C", "time_step", "u_in", "u_out", "breath_id"]
X_train = train_df[feature_cols].copy()
y_train = train_df["pressure"]

# Fill any remaining NaNs in the target (defensive)
y_train = y_train.fillna(0)

X_test = test_df[feature_cols].copy()


def add_engineered_features(df):
    df["R_C"] = df["R"] * df["C"]
    df["u_in_sq"] = df["u_in"] ** 2
    df["time_sq"] = df["time_step"] ** 2
    df["u_in_time"] = df["u_in"] * df["time_step"]
    df["breath_log"] = np.log1p(df["breath_id"])
    df["u_out_u_in"] = df["u_out"] * df["u_in"]
    df["R_C_u_in"] = df["R"] * df["C"] * df["u_in"]

    df["cum_u_in"] = df.groupby("breath_id")["u_in"].cumsum()
    df["cum_u_out"] = df.groupby("breath_id")["u_out"].cumsum()
    df["diff_u_in"] = df.groupby("breath_id")["u_in"].diff().fillna(0)
    df["diff_u_out"] = df.groupby("breath_id")["u_out"].diff().fillna(0)

    grp = df.groupby("breath_id")
    df["breath_max_u_in"] = grp["u_in"].transform("max")
    df["breath_min_u_in"] = grp["u_in"].transform("min")
    df["breath_mean_u_in"] = grp["u_in"].transform("mean")
    df["breath_max_time"] = grp["time_step"].transform("max")
    df["cum_time"] = grp["time_step"].cumsum()

    df["R_div_C"] = df["R"] / df["C"]
    df["breath_len"] = grp["time_step"].transform("size")
    df["time_norm"] = df["time_step"] / df["breath_max_time"]

    df["R_sq"] = df["R"] ** 2
    df["C_sq"] = df["C"] ** 2
    df["u_in_R_ratio"] = df["u_in"] / (df["R"] + 1e-6)
    df["u_in_C_ratio"] = df["u_in"] / (df["C"] + 1e-6)
    df["time_u_in_ratio"] = df["time_step"] / (df["u_in"] + 1e-6)
    df["R_plus_C"] = df["R"] + df["C"]
    df["R_minus_C"] = df["R"] - df["C"]

    return df


X_train = add_engineered_features(X_train)
X_test = add_engineered_features(X_test)

# Cast to float32 for stable handling by HistGradientBoostingRegressor
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)




In [3]:
y_train_log = np.log1p(y_train)

# Ensure no NaNs after log‑transform (defensive)
y_train_log = np.where(np.isfinite(y_train_log), y_train_log, 0)

model = HistGradientBoostingRegressor(
    max_depth=20,  # slightly deeper trees for richer feature interactions
    learning_rate=0.02,  # finer learning steps
    max_iter=2500,  # more iterations to fully utilise the deeper trees
    l2_regularization=0.1,  # modest regularisation to curb over‑fit
    random_state=42,
)

model.fit(X_train, y_train_log)

test_pred = np.expm1(model.predict(X_test))

# Clamp predictions to zero where no inspiratory flow occurs
test_pred = np.where(X_test["u_in"] == 0, 0.0, test_pred)

submission = pd.DataFrame({"id": test_df["id"], "pressure": test_pred})
submission.to_csv("submission.csv", index=False)




/usr/local/lib/python3.11/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
print(submission.head())

   id   pressure
0   1   5.767898
1   2   5.699599
2   3   6.450472
3   4   8.340371
4   5  11.795185
